In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv("jobs_feature_engineered.csv")

In [3]:
df=df1.copy()

In [4]:
# Keep only rows where contract type is known
contract_df = df[df['contract_type'].notna()].copy()

In [5]:
df.shape

(7269, 74)

In [6]:
contract_df.shape

(256, 74)

In [7]:
contract_df['contract_type'].value_counts()

contract_type
permanent    147
contract     109
Name: count, dtype: int64

In [8]:
contract_df['company_name'].nunique()

187

In [9]:
contract_df['source'].value_counts()

source
Adzuna    256
Name: count, dtype: int64

In [10]:
core_features = ['location_clean','category','standard_role','contract_time',
    'posting_year','posting_month', 'title_word_count','title_length',
    'title_has_senior', 'title_has_lead','title_has_manager','title_has_intern',
    'title_has_junior','title_has_principal','title_has_staff','title_has_associate']

In [11]:
print("Number of core features:", len(core_features))
print(core_features)

Number of core features: 16
['location_clean', 'category', 'standard_role', 'contract_time', 'posting_year', 'posting_month', 'title_word_count', 'title_length', 'title_has_senior', 'title_has_lead', 'title_has_manager', 'title_has_intern', 'title_has_junior', 'title_has_principal', 'title_has_staff', 'title_has_associate']


In [12]:
from sklearn.model_selection import StratifiedGroupKFold

In [13]:
group_splitter = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)

In [14]:
splits = list(group_splitter.split(contract_df,contract_df['contract_type'],groups=contract_df['company_name']))

In [15]:
# Pre-committed final holdout: Fold 1
dev_idx, holdout_idx = splits[0]

In [16]:
dev_data = contract_df.iloc[dev_idx].copy()
holdout_data = contract_df.iloc[holdout_idx].copy()

In [17]:
X = dev_data[core_features].copy()
y = (dev_data['contract_type'] == 'contract').astype(int)

In [18]:
X_holdout = holdout_data[core_features].copy()
y_holdout = (holdout_data['contract_type'] == 'contract').astype(int)

In [19]:
print("Development shape:", X.shape)
print("Holdout shape:", X_holdout.shape)

Development shape: (205, 16)
Holdout shape: (51, 16)


In [20]:
print(y.value_counts())
print(y.value_counts(normalize=True))

contract_type
0    118
1     87
Name: count, dtype: int64
contract_type
0    0.57561
1    0.42439
Name: proportion, dtype: float64


In [21]:
print(y_holdout.value_counts())
print(y_holdout.value_counts(normalize=True))


contract_type
0    29
1    22
Name: count, dtype: int64
contract_type
0    0.568627
1    0.431373
Name: proportion, dtype: float64


In [22]:
print("Development:", dev_data['company_name'].nunique())
print("Holdout:", holdout_data['company_name'].nunique())

Development: 151
Holdout: 36


In [23]:
print("\nCompany overlap:")
print(len(set(dev_data['company_name']) & set(holdout_data['company_name'])))


Company overlap:
0


In [24]:
# Check that no target/leakage columns are present in the model features

excluded_features = ['contract_type','job_level','job_type','job_level_clean',
    'source','company_name','title', 'city','state', 'title_has_director']

In [25]:
present_excluded = [col for col in excluded_features
    if col in X.columns or col in X_holdout.columns]

In [26]:
print("Excluded columns present in X/X_holdout:")
print(present_excluded)

Excluded columns present in X/X_holdout:
[]


In [27]:
print("\nFeature count:", len(X.columns))


Feature count: 16


In [28]:
print("\nMissing values in development:")
print(X.isna().sum()[X.isna().sum() > 0])

print("\nMissing values in holdout:")
print(X_holdout.isna().sum()[X_holdout.isna().sum() > 0])


Missing values in development:
contract_time    55
dtype: int64

Missing values in holdout:
contract_time    7
dtype: int64


In [29]:
categorical_features = ['location_clean','category','standard_role', 'contract_time']

In [30]:
numeric_features = ['posting_year','posting_month','title_word_count', 'title_length',
    'title_has_senior','title_has_lead','title_has_manager','title_has_intern',
    'title_has_junior','title_has_principal','title_has_staff', 'title_has_associate']

In [31]:
print("Categorical features:", len(categorical_features))
print(categorical_features)

Categorical features: 4
['location_clean', 'category', 'standard_role', 'contract_time']


In [32]:
print("\nNumeric features:", len(numeric_features))
print(numeric_features)


Numeric features: 12
['posting_year', 'posting_month', 'title_word_count', 'title_length', 'title_has_senior', 'title_has_lead', 'title_has_manager', 'title_has_intern', 'title_has_junior', 'title_has_principal', 'title_has_staff', 'title_has_associate']


In [33]:
print("\nData types:")
print(X[categorical_features + numeric_features].dtypes)


Data types:
location_clean         object
category               object
standard_role          object
contract_time          object
posting_year            int64
posting_month           int64
title_word_count        int64
title_length            int64
title_has_senior        int64
title_has_lead          int64
title_has_manager       int64
title_has_intern        int64
title_has_junior        int64
title_has_principal     int64
title_has_staff         int64
title_has_associate     int64
dtype: object


In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [35]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])


In [36]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

In [37]:
preprocessor = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_features),
    ('numeric', numeric_pipeline, numeric_features)])

In [38]:
print("Preprocessor created successfully.")
print(preprocessor)

Preprocessor created successfully.
ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['location_clean', 'category', 'standard_role',
                                  'contract_time']),
                                ('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['posting_year', 'posting_month',
                                  'title_word_count', 'ti

In [39]:
# LogisticRegression

In [40]:
from sklearn.linear_model import LogisticRegression

In [41]:
lr_model = Pipeline([('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=2000, random_state=42))])

In [42]:
print(lr_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Missing',
                                                                                 strategy='constant')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['location_clean', 'category',
                                                   'standard_role',
                                                   'contract_time']),
                                                 ('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   Simp

In [43]:
from sklearn.model_selection import cross_validate

In [44]:
cv = StratifiedGroupKFold( n_splits=5,shuffle=True, random_state=42)

In [45]:
scoring = {'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'}

In [46]:
lr_cv = cross_validate(lr_model,X, y,cv=cv,groups=dev_data['company_name'],
    scoring=scoring,return_train_score=False,error_score='raise')

In [47]:
for i in range(5):
    print( f"Fold {i+1}: "
        f"Accuracy={lr_cv['test_accuracy'][i]:.4f}, "
        f"Precision={lr_cv['test_precision'][i]:.4f}, "
        f"Recall={lr_cv['test_recall'][i]:.4f}, "
        f"F1={lr_cv['test_f1'][i]:.4f}, "
        f"ROC-AUC={lr_cv['test_roc_auc'][i]:.4f}" )

Fold 1: Accuracy=0.6585, Precision=0.5789, Recall=0.6471, F1=0.6111, ROC-AUC=0.7365
Fold 2: Accuracy=0.5250, Precision=0.3750, Recall=0.1765, F1=0.2400, ROC-AUC=0.5473
Fold 3: Accuracy=0.7000, Precision=0.6316, Recall=0.7059, F1=0.6667, ROC-AUC=0.7570
Fold 4: Accuracy=0.8571, Precision=0.8750, Recall=0.7778, F1=0.8235, ROC-AUC=0.9051
Fold 5: Accuracy=0.7143, Precision=0.6500, Recall=0.7222, F1=0.6842, ROC-AUC=0.7894


In [48]:
print("\nMean ± Std:")
for metric in scoring:
    scores = lr_cv[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")


Mean ± Std:
accuracy: 0.6910 ± 0.1066
precision: 0.6221 ± 0.1598
recall: 0.6059 ± 0.2187
f1: 0.6051 ± 0.1955
roc_auc: 0.7471 ± 0.1156


In [49]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

In [50]:
dummy_model = DummyClassifier( strategy='most_frequent')

In [51]:

dummy_scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer( precision_score, zero_division=0 ),
    'recall': make_scorer(recall_score, zero_division=0 ),
    'f1': make_scorer( f1_score, zero_division=0),
    'roc_auc': 'roc_auc'}


In [52]:
dummy_cv = cross_validate(dummy_model, X,y, cv=cv, groups=dev_data['company_name'],scoring=dummy_scoring,return_train_score=False, error_score='raise')

In [53]:
print("Dummy baseline:")
for metric in dummy_scoring:
    scores = dummy_cv[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")

Dummy baseline:
accuracy: 0.5756 ± 0.0051
precision: 0.0000 ± 0.0000
recall: 0.0000 ± 0.0000
f1: 0.0000 ± 0.0000
roc_auc: 0.5000 ± 0.0000


In [54]:
from sklearn.tree import DecisionTreeClassifier

In [55]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [56]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

In [57]:
dt_model = Pipeline([('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier( random_state=42))])

In [58]:
dt_cv = cross_validate(dt_model,X, y,cv=cv,groups=dev_data['company_name'],
    scoring=scoring, return_train_score=False, error_score='raise')

In [59]:
print("Decision Tree fold-wise results:")

for i in range(5):
    print(f"Fold {i+1}: "
        f"Accuracy={dt_cv['test_accuracy'][i]:.4f}, "
        f"Precision={dt_cv['test_precision'][i]:.4f}, "
        f"Recall={dt_cv['test_recall'][i]:.4f}, "
        f"F1={dt_cv['test_f1'][i]:.4f}, "
        f"ROC-AUC={dt_cv['test_roc_auc'][i]:.4f}" )

Decision Tree fold-wise results:
Fold 1: Accuracy=0.7073, Precision=0.6923, Recall=0.5294, F1=0.6000, ROC-AUC=0.6814
Fold 2: Accuracy=0.6500, Precision=0.5714, Recall=0.7059, F1=0.6316, ROC-AUC=0.6573
Fold 3: Accuracy=0.5250, Precision=0.4375, Recall=0.4118, F1=0.4242, ROC-AUC=0.5102
Fold 4: Accuracy=0.7143, Precision=0.6364, Recall=0.7778, F1=0.7000, ROC-AUC=0.7222
Fold 5: Accuracy=0.7381, Precision=0.6842, Recall=0.7222, F1=0.7027, ROC-AUC=0.7361


In [60]:
print("\nMean ± Std:")
for metric in scoring:
    scores = dt_cv[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")


Mean ± Std:
accuracy: 0.6669 ± 0.0766
precision: 0.6044 ± 0.0939
recall: 0.6294 ± 0.1370
f1: 0.6117 ± 0.1017
roc_auc: 0.6614 ± 0.0807


In [61]:
# Random Forest 

In [62]:
from sklearn.ensemble import RandomForestClassifier

In [63]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [64]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

In [65]:
rf_model = Pipeline([('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=300, random_state=42,n_jobs=-1 ))])

In [66]:
rf_cv = cross_validate( rf_model,X, y,cv=cv,groups=dev_data['company_name'],scoring=scoring,return_train_score=False, error_score='raise')

In [67]:
print("Random Forest fold-wise results:")

for i in range(5):
    print(
        f"Fold {i+1}: "
        f"Accuracy={rf_cv['test_accuracy'][i]:.4f}, "
        f"Precision={rf_cv['test_precision'][i]:.4f}, "
        f"Recall={rf_cv['test_recall'][i]:.4f}, "
        f"F1={rf_cv['test_f1'][i]:.4f}, "
        f"ROC-AUC={rf_cv['test_roc_auc'][i]:.4f}")

Random Forest fold-wise results:
Fold 1: Accuracy=0.7317, Precision=0.7143, Recall=0.5882, F1=0.6452, ROC-AUC=0.8370
Fold 2: Accuracy=0.6000, Precision=0.5294, Recall=0.5294, F1=0.5294, ROC-AUC=0.6969
Fold 3: Accuracy=0.6000, Precision=0.5333, Recall=0.4706, F1=0.5000, ROC-AUC=0.7008
Fold 4: Accuracy=0.7857, Precision=0.8000, Recall=0.6667, F1=0.7273, ROC-AUC=0.8750
Fold 5: Accuracy=0.7619, Precision=0.7857, Recall=0.6111, F1=0.6875, ROC-AUC=0.8194


In [68]:
print("\nMean ± Std:")
for metric in scoring:
    scores = rf_cv[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")


Mean ± Std:
accuracy: 0.6959 ± 0.0801
precision: 0.6725 ± 0.1189
recall: 0.5732 ± 0.0676
f1: 0.6179 ± 0.0886
roc_auc: 0.7858 ± 0.0733


In [69]:
from xgboost import XGBClassifier

In [70]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [71]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

In [72]:
xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier( n_estimators=300, max_depth=4,learning_rate=0.08,
        subsample=0.8,colsample_bytree=0.8,objective='binary:logistic',
        eval_metric='logloss', random_state=42, n_jobs=-1))])

In [73]:
xgb_cv = cross_validate(xgb_model,X,y,cv=cv,
    groups=dev_data['company_name'],
    scoring=scoring,
    return_train_score=False,
    error_score='raise')


In [74]:
print("XGBoost fold-wise results:")

for i in range(5):
    print(
        f"Fold {i+1}: "
        f"Accuracy={xgb_cv['test_accuracy'][i]:.4f}, "
        f"Precision={xgb_cv['test_precision'][i]:.4f}, "
        f"Recall={xgb_cv['test_recall'][i]:.4f}, "
        f"F1={xgb_cv['test_f1'][i]:.4f}, "
        f"ROC-AUC={xgb_cv['test_roc_auc'][i]:.4f}" )

XGBoost fold-wise results:
Fold 1: Accuracy=0.7805, Precision=0.7222, Recall=0.7647, F1=0.7429, ROC-AUC=0.8150
Fold 2: Accuracy=0.7500, Precision=0.7059, Recall=0.7059, F1=0.7059, ROC-AUC=0.7417
Fold 3: Accuracy=0.5500, Precision=0.4615, Recall=0.3529, F1=0.4000, ROC-AUC=0.6726
Fold 4: Accuracy=0.8095, Precision=0.7778, Recall=0.7778, F1=0.7778, ROC-AUC=0.8634
Fold 5: Accuracy=0.8095, Precision=0.8571, Recall=0.6667, F1=0.7500, ROC-AUC=0.8495


In [75]:
print("\nMean ± Std:")

for metric in scoring:
    scores = xgb_cv[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")


Mean ± Std:
accuracy: 0.7399 ± 0.0975
precision: 0.7049 ± 0.1327
recall: 0.6536 ± 0.1556
f1: 0.6753 ± 0.1396
roc_auc: 0.7884 ± 0.0716


In [76]:
inner_cv = StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42)

In [77]:
print("Inner CV created.")
print("Number of folds:", inner_cv.n_splits)
print("Shuffle:", inner_cv.shuffle)
print("Random state:", inner_cv.random_state)

Inner CV created.
Number of folds: 4
Shuffle: True
Random state: 42


In [78]:
param_distributions = {
    'model__n_estimators': [100, 200, 300, 400, 500],
    'model__max_depth': [2, 3, 4, 5, 6],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.08, 0.10],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5],
    'model__gamma': [0, 0.1, 0.3],
    'model__reg_alpha': [0, 0.1, 0.5],
    'model__reg_lambda': [1, 2, 5]}

print("Number of hyperparameters:", len(param_distributions))

Number of hyperparameters: 9


In [79]:
print("\nHyperparameter search space:")
for parameter, values in param_distributions.items():
    print(f"{parameter}: {values}")


Hyperparameter search space:
model__n_estimators: [100, 200, 300, 400, 500]
model__max_depth: [2, 3, 4, 5, 6]
model__learning_rate: [0.01, 0.03, 0.05, 0.08, 0.1]
model__subsample: [0.7, 0.8, 0.9, 1.0]
model__colsample_bytree: [0.7, 0.8, 0.9, 1.0]
model__min_child_weight: [1, 3, 5]
model__gamma: [0, 0.1, 0.3]
model__reg_alpha: [0, 0.1, 0.5]
model__reg_lambda: [1, 2, 5]


In [80]:
from sklearn.model_selection import RandomizedSearchCV

In [81]:
xgb_search = RandomizedSearchCV( estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=40, scoring='f1',cv=inner_cv,
    random_state=42,n_jobs=-1,return_train_score=False,
    error_score='raise')

In [82]:
xgb_search.fit(X,y, groups=dev_data['company_name'])

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': [0.7, 0.8, ...], 'model__gamma': [0, 0.1, ...], 'model__learning_rate': [0.01, 0.03, ...], 'model__max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies 

In [83]:
print("Best CV F1:", round(xgb_search.best_score_, 4))

print("\nBest parameters:")
print(xgb_search.best_params_)

Best CV F1: 0.656

Best parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 2, 'model__reg_alpha': 0.1, 'model__n_estimators': 400, 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.1, 'model__gamma': 0.1, 'model__colsample_bytree': 0.8}


In [84]:
models = {'Logistic Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'Tuned XGBoost': xgb_search.best_estimator_}

In [85]:
comparison = {}

for name, model in models.items():
    result = cross_validate(model,X, y, cv=inner_cv,
        groups=dev_data['company_name'],scoring=scoring,
        return_train_score=False,error_score='raise')

    comparison[name] = {metric: result[f'test_{metric}'].mean()for metric in scoring}

In [86]:
print("Clean inner-CV comparison:\n")
for name, metrics in comparison.items():
    print(name)
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    print()

Clean inner-CV comparison:

Logistic Regression
  accuracy: 0.6689
  precision: 0.6391
  recall: 0.5298
  f1: 0.5763
  roc_auc: 0.7451

Decision Tree
  accuracy: 0.6491
  precision: 0.6072
  recall: 0.5768
  f1: 0.5792
  roc_auc: 0.6403

Random Forest
  accuracy: 0.6879
  precision: 0.6635
  recall: 0.5519
  f1: 0.6013
  roc_auc: 0.7428

XGBoost
  accuracy: 0.6929
  precision: 0.6600
  recall: 0.5747
  f1: 0.6141
  roc_auc: 0.7482

Tuned XGBoost
  accuracy: 0.7220
  precision: 0.7023
  recall: 0.5985
  f1: 0.6453
  roc_auc: 0.7583



In [87]:
from sklearn.model_selection import GridSearchCV

In [88]:
lr_param_grid = {'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__solver': ['lbfgs', 'liblinear'],
    'model__class_weight': [None, 'balanced']}

print("Number of parameter combinations:", 24)

for parameter, values in lr_param_grid.items():
    print(f"{parameter}: {values}")

Number of parameter combinations: 24
model__C: [0.001, 0.01, 0.1, 1, 10, 100]
model__solver: ['lbfgs', 'liblinear']
model__class_weight: [None, 'balanced']


In [89]:
lr_search = GridSearchCV( estimator=lr_model,param_grid=lr_param_grid,
    scoring=scoring, refit='f1',cv=inner_cv, n_jobs=-1,return_train_score=False,
    error_score='raise')

In [90]:
lr_search.fit( X, y,groups=dev_data['company_name'])

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None, 'balanced'], 'model__solver': ['lbfgs', 'liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedGro... shuffle=True)
,"verbose verbose: intCo

In [91]:
print("Best CV F1:", round(lr_search.best_score_, 4))
print("\nBest parameters:")
print(lr_search.best_params_)

Best CV F1: 0.6425

Best parameters:
{'model__C': 100, 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}


In [92]:
best_index = lr_search.best_index_

print("Tuned Logistic Regression metrics:")
print(f"Accuracy:  {lr_search.cv_results_['mean_test_accuracy'][best_index]:.4f} ± {lr_search.cv_results_['std_test_accuracy'][best_index]:.4f}")
print(f"Precision: {lr_search.cv_results_['mean_test_precision'][best_index]:.4f} ± {lr_search.cv_results_['std_test_precision'][best_index]:.4f}")
print(f"Recall:    {lr_search.cv_results_['mean_test_recall'][best_index]:.4f} ± {lr_search.cv_results_['std_test_recall'][best_index]:.4f}")
print(f"F1:        {lr_search.cv_results_['mean_test_f1'][best_index]:.4f} ± {lr_search.cv_results_['std_test_f1'][best_index]:.4f}")
print(f"ROC-AUC:   {lr_search.cv_results_['mean_test_roc_auc'][best_index]:.4f} ± {lr_search.cv_results_['std_test_roc_auc'][best_index]:.4f}")

Tuned Logistic Regression metrics:
Accuracy:  0.6986 ± 0.0795
Precision: 0.6550 ± 0.0929
Recall:    0.6461 ± 0.1314
F1:        0.6425 ± 0.0897
ROC-AUC:   0.7940 ± 0.0466


In [93]:
dt_param_grid = { 'model__max_depth': [2, 3, 4, 5, 6, 8, 10, None],
    'model__min_samples_split': [2, 5, 10, 15, 20],
    'model__min_samples_leaf': [1, 2, 4, 6, 10],
    'model__criterion': ['gini', 'entropy']}

total_combinations = 8 * 5 * 5 * 2

print("Number of parameter combinations:", total_combinations)

for parameter, values in dt_param_grid.items():
    print(f"{parameter}: {values}")

Number of parameter combinations: 400
model__max_depth: [2, 3, 4, 5, 6, 8, 10, None]
model__min_samples_split: [2, 5, 10, 15, 20]
model__min_samples_leaf: [1, 2, 4, 6, 10]
model__criterion: ['gini', 'entropy']


In [94]:
dt_search = GridSearchCV(estimator=dt_model, param_grid=dt_param_grid,scoring=scoring, refit='f1',
    cv=inner_cv, n_jobs=-1,return_train_score=False, error_score='raise')

In [95]:
dt_search.fit( X, y, groups=dev_data['company_name'])

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedGro...

In [96]:
print("Best CV F1:", round(dt_search.best_score_, 4))

print("\nBest parameters:")
print(dt_search.best_params_)

Best CV F1: 0.5915

Best parameters:
{'model__criterion': 'entropy', 'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}


In [97]:
best_index = dt_search.best_index_

print("Tuned Decision Tree metrics:")
print(f"Accuracy:  {dt_search.cv_results_['mean_test_accuracy'][best_index]:.4f} ± {dt_search.cv_results_['std_test_accuracy'][best_index]:.4f}")
print(f"Precision: {dt_search.cv_results_['mean_test_precision'][best_index]:.4f} ± {dt_search.cv_results_['std_test_precision'][best_index]:.4f}")
print(f"Recall:    {dt_search.cv_results_['mean_test_recall'][best_index]:.4f} ± {dt_search.cv_results_['std_test_recall'][best_index]:.4f}")
print(f"F1:        {dt_search.cv_results_['mean_test_f1'][best_index]:.4f} ± {dt_search.cv_results_['std_test_f1'][best_index]:.4f}")
print(f"ROC-AUC:   {dt_search.cv_results_['mean_test_roc_auc'][best_index]:.4f} ± {dt_search.cv_results_['std_test_roc_auc'][best_index]:.4f}")

Tuned Decision Tree metrics:
Accuracy:  0.6923 ± 0.0689
Precision: 0.7058 ± 0.1451
Recall:    0.5281 ± 0.1010
F1:        0.5915 ± 0.0820
ROC-AUC:   0.7092 ± 0.0858


In [98]:
rf_param_grid = {'model__n_estimators': [200, 300, 500],
    'model__max_depth': [None, 5, 10, 15, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', None],
    'model__class_weight': [None, 'balanced']}

total_combinations = ( 3 * 5 * 3 * 3 * 3 * 2)

print("Number of parameter combinations:", total_combinations)

for parameter, values in rf_param_grid.items():
    print(f"{parameter}: {values}")

Number of parameter combinations: 810
model__n_estimators: [200, 300, 500]
model__max_depth: [None, 5, 10, 15, 20]
model__min_samples_split: [2, 5, 10]
model__min_samples_leaf: [1, 2, 4]
model__max_features: ['sqrt', 'log2', None]
model__class_weight: [None, 'balanced']


In [99]:
rf_search = GridSearchCV(estimator=rf_model, param_grid=rf_param_grid,
    scoring=scoring,refit='f1', cv=inner_cv,n_jobs=-1,  return_train_score=False,error_score='raise')

In [100]:
rf_search.fit( X, y,groups=dev_data['company_name'])

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__class_weight': [None, 'balanced'], 'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': [1, 2, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",S

In [101]:
print("Best CV F1:", round(rf_search.best_score_, 4))
print("\nBest parameters:")
print(rf_search.best_params_)

Best CV F1: 0.6329

Best parameters:
{'model__class_weight': 'balanced', 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 10, 'model__n_estimators': 300}


In [102]:
best_index = rf_search.best_index_
print("Tuned Random Forest metrics:")
print(f"Accuracy:  {rf_search.cv_results_['mean_test_accuracy'][best_index]:.4f} ± {rf_search.cv_results_['std_test_accuracy'][best_index]:.4f}")
print(f"Precision: {rf_search.cv_results_['mean_test_precision'][best_index]:.4f} ± {rf_search.cv_results_['std_test_precision'][best_index]:.4f}")
print(f"Recall:    {rf_search.cv_results_['mean_test_recall'][best_index]:.4f} ± {rf_search.cv_results_['std_test_recall'][best_index]:.4f}")
print(f"F1:        {rf_search.cv_results_['mean_test_f1'][best_index]:.4f} ± {rf_search.cv_results_['std_test_f1'][best_index]:.4f}")
print(f"ROC-AUC:   {rf_search.cv_results_['mean_test_roc_auc'][best_index]:.4f} ± {rf_search.cv_results_['std_test_roc_auc'][best_index]:.4f}")

Tuned Random Forest metrics:
Accuracy:  0.6976 ± 0.0547
Precision: 0.6654 ± 0.0872
Recall:    0.6098 ± 0.0459
F1:        0.6329 ± 0.0493
ROC-AUC:   0.7537 ± 0.0292


In [103]:
# tune Xgboost

In [104]:
xgb_search = RandomizedSearchCV(estimator=xgb_model,param_distributions=param_distributions,
    n_iter=40,scoring=scoring, refit='f1', cv=inner_cv,random_state=42,n_jobs=-1,
    return_train_score=False, error_score='raise')

In [105]:
xgb_search.fit( X,y, groups=dev_data['company_name'])

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': [0.7, 0.8, ...], 'model__gamma': [0, 0.1, ...], 'model__learning_rate': [0.01, 0.03, ...], 'model__max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'accuracy': 'accuracy', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the s

In [106]:
print("Best CV F1:", round(xgb_search.best_score_, 4))
print("\nBest parameters:")
print(xgb_search.best_params_)

Best CV F1: 0.656

Best parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 2, 'model__reg_alpha': 0.1, 'model__n_estimators': 400, 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.1, 'model__gamma': 0.1, 'model__colsample_bytree': 0.8}


In [107]:
best_index = xgb_search.best_index_
print("Tuned XGBoost metrics:")
print(f"Accuracy:  {xgb_search.cv_results_['mean_test_accuracy'][best_index]:.4f} ± {xgb_search.cv_results_['std_test_accuracy'][best_index]:.4f}")
print(f"Precision: {xgb_search.cv_results_['mean_test_precision'][best_index]:.4f} ± {xgb_search.cv_results_['std_test_precision'][best_index]:.4f}")
print(f"Recall:    {xgb_search.cv_results_['mean_test_recall'][best_index]:.4f} ± {xgb_search.cv_results_['std_test_recall'][best_index]:.4f}")
print(f"F1:        {xgb_search.cv_results_['mean_test_f1'][best_index]:.4f} ± {xgb_search.cv_results_['std_test_f1'][best_index]:.4f}")
print(f"ROC-AUC:   {xgb_search.cv_results_['mean_test_roc_auc'][best_index]:.4f} ± {xgb_search.cv_results_['std_test_roc_auc'][best_index]:.4f}")

Tuned XGBoost metrics:
Accuracy:  0.7319 ± 0.0488
Precision: 0.7128 ± 0.0599
Recall:    0.6098 ± 0.0911
F1:        0.6560 ± 0.0775
ROC-AUC:   0.7648 ± 0.0284


In [108]:
# Feature importance from the selected XGBoost model
selected_model = xgb_search.best_estimator_

In [109]:
xgb_estimator = selected_model.named_steps['model']
preprocessor_fitted = selected_model.named_steps['preprocessor']

In [110]:
# Get transformed feature names after preprocessing
transformed_features = preprocessor_fitted.get_feature_names_out()

In [111]:
# Get XGBoost importance values
importance_values = xgb_estimator.feature_importances_

In [112]:
# Create feature-importance table
importance_df = pd.DataFrame({
    'transformed_feature': transformed_features,
    'importance': importance_values})


In [113]:
# Map each transformed feature back to its original feature
def get_original_feature(feature_name):
    for feature in categorical_features:
        if feature_name.startswith(f'categorical__{feature}'):
            return feature

    for feature in numeric_features:
        if feature_name == f'numeric__{feature}':
            return feature

    return feature_name


In [114]:
importance_df['original_feature'] = (importance_df['transformed_feature'].apply(get_original_feature))

In [115]:
# Aggregate importance of one-hot encoded columns
aggregated_importance = (importance_df
    .groupby('original_feature', as_index=False)['importance']
    .sum()
    .sort_values('importance', ascending=False))

In [116]:
print("XGBoost importance type:")
print(xgb_estimator.importance_type)

XGBoost importance type:
None


In [117]:
print("\nTop transformed features:")
print(importance_df.sort_values('importance', ascending=False).head(20) .to_string(index=False))


Top transformed features:
                                 transformed_feature  importance original_feature
         categorical__standard_role_Business Analyst    0.073930    standard_role
     categorical__location_clean_Mumbai, Maharashtra    0.064472   location_clean
        categorical__standard_role_Software Engineer    0.063816    standard_role
                   categorical__location_clean_India    0.059930   location_clean
                               numeric__posting_year    0.050068     posting_year
           categorical__standard_role_Java Developer    0.048826    standard_role
                  categorical__contract_time_Missing    0.045075    contract_time
                              numeric__posting_month    0.043297    posting_month
             categorical__standard_role_Data Analyst    0.040115    standard_role
        categorical__standard_role_Backend Developer    0.037250    standard_role
     categorical__location_clean_Chennai, Tamil Nadu    0.035367   loca

In [118]:
print("\nAggregated importance by original feature:")
print( aggregated_importance.to_string(index=False))
print("\nImportance sum:", round(aggregated_importance['importance'].sum(), 4))


Aggregated importance by original feature:
   original_feature  importance
      standard_role    0.457017
     location_clean    0.214218
      contract_time    0.079770
           category    0.058271
       posting_year    0.050068
      posting_month    0.043297
   title_has_senior    0.033933
       title_length    0.032484
   title_word_count    0.030941
title_has_associate    0.000000
   title_has_intern    0.000000
   title_has_junior    0.000000
     title_has_lead    0.000000
  title_has_manager    0.000000
title_has_principal    0.000000
    title_has_staff    0.000000

Importance sum: 1.0


In [119]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone


In [120]:
importance_results = []

for fold, (train_idx, valid_idx) in enumerate(
    inner_cv.split(X, y, groups=dev_data['company_name'] ),
    start=1):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    # Create a fresh copy of the tuned model
    model = clone(xgb_search.best_estimator_)

    # Fit only on this fold's training companies
    model.fit(X_train, y_train)

    perm = permutation_importance(model,X_valid,y_valid,scoring='f1',n_repeats=10,
        random_state=42, n_jobs=-1)

    for feature, importance in zip(X.columns, perm.importances_mean):
        importance_results.append({ 'fold': fold,
            'feature': feature,
            'importance': importance })

permutation_df = pd.DataFrame(importance_results)



In [121]:
permutation_summary = ( permutation_df
    .groupby('feature')['importance']
    .agg(['mean', 'std'])
    .sort_values('mean', ascending=False))

print("Corrected group-aware permutation importance:")
print(permutation_summary)

Corrected group-aware permutation importance:
                         mean       std
feature                                
posting_month        0.104224  0.057498
location_clean       0.091009  0.015721
contract_time        0.065467  0.068360
title_length         0.060622  0.068567
posting_year         0.055969  0.062281
standard_role        0.038885  0.072704
title_word_count     0.031909  0.031898
category             0.006772  0.004781
title_has_senior     0.004130  0.001530
title_has_associate  0.000000  0.000000
title_has_intern     0.000000  0.000000
title_has_junior     0.000000  0.000000
title_has_lead       0.000000  0.000000
title_has_manager    0.000000  0.000000
title_has_principal  0.000000  0.000000
title_has_staff      0.000000  0.000000


In [122]:
selected_model = clone(xgb_search.best_estimator_)
selected_model.fit(X, y)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [123]:
print("Final model fitted successfully.")
print("Training rows:", X.shape[0])
print("Training companies:", dev_data['company_name'].nunique())
print("\nSelected parameters:")
print(xgb_search.best_params_)

Final model fitted successfully.
Training rows: 205
Training companies: 151

Selected parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 2, 'model__reg_alpha': 0.1, 'model__n_estimators': 400, 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.1, 'model__gamma': 0.1, 'model__colsample_bytree': 0.8}


In [124]:
check_model = clone(xgb_search.best_estimator_)

In [125]:
check_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [126]:
check_preprocessor = check_model.named_steps['preprocessor']

In [127]:
check_numeric_pipeline = ( check_preprocessor.named_transformers_['numeric'])


In [128]:
check_imputer = check_numeric_pipeline.named_steps['imputer']
check_scaler = check_numeric_pipeline.named_steps['scaler']

In [129]:
print("X shape:", X.shape)
print("\nImputer statistics shape:")
print(check_imputer.statistics_.shape)
print("\nScaler samples seen:")
print(check_scaler.n_samples_seen_)
print("\nExpected samples:")
print(len(X))
print("\nTransformed X shape:")
print(check_preprocessor.transform(X).shape)

X shape: (205, 16)

Imputer statistics shape:
(12,)

Scaler samples seen:
205.0

Expected samples:
205

Transformed X shape:
(205, 83)


In [130]:
# Reduced features

In [131]:
zero_importance_features = ['title_has_associate','title_has_intern','title_has_junior',
    'title_has_lead','title_has_manager','title_has_principal','title_has_staff']

In [132]:
reduced_categorical_features = [ feature for feature in categorical_features
                                if feature not in zero_importance_features]


In [133]:
reduced_numeric_features = [ feature for feature in numeric_features
                            if feature not in zero_importance_features]


In [134]:
reduced_features = (reduced_categorical_features + reduced_numeric_features)

In [135]:
print("Original feature count:", len(core_features))
print("Reduced feature count:", len(reduced_features))
print("\nRemoved features:")
print(zero_importance_features)
print("\nRemaining features:")
print(reduced_features)

Original feature count: 16
Reduced feature count: 9

Removed features:
['title_has_associate', 'title_has_intern', 'title_has_junior', 'title_has_lead', 'title_has_manager', 'title_has_principal', 'title_has_staff']

Remaining features:
['location_clean', 'category', 'standard_role', 'contract_time', 'posting_year', 'posting_month', 'title_word_count', 'title_length', 'title_has_senior']


In [136]:
reduced_categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Missing' )),
                                         ('encoder', OneHotEncoder( handle_unknown='ignore' ))])

In [137]:
reduced_numeric_pipeline = Pipeline([('imputer', SimpleImputer( strategy='median')),
    ('scaler', StandardScaler())])

In [138]:
reduced_preprocessor = ColumnTransformer([('categorical',reduced_categorical_pipeline, reduced_categorical_features),
                    ('numeric',reduced_numeric_pipeline, reduced_numeric_features)])

In [139]:
best_params = {key.replace('model__', ''): value
    for key, value in xgb_search.best_params_.items()}

In [140]:
reduced_xgb_model = Pipeline([('preprocessor', reduced_preprocessor),
    ('model', XGBClassifier(**best_params,objective='binary:logistic',
        eval_metric='logloss',random_state=42, n_jobs=-1 ))])

In [141]:
print("Reduced XGBoost pipeline created.")
print("Number of features:", len(reduced_features))
print("\nXGBoost parameters:")
print(best_params)

Reduced XGBoost pipeline created.
Number of features: 9

XGBoost parameters:
{'subsample': 0.8, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 2, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.8}


In [142]:
reduced_cv = cross_validate(reduced_xgb_model, X[reduced_features], y,
    cv=inner_cv,groups=dev_data['company_name'],scoring=scoring,
    return_train_score=False,error_score='raise')

In [143]:
print("Reduced XGBoost fold-wise results:")
for i in range(4):
    print(
        f"Fold {i+1}: "
        f"Accuracy={reduced_cv['test_accuracy'][i]:.4f}, "
        f"Precision={reduced_cv['test_precision'][i]:.4f}, "
        f"Recall={reduced_cv['test_recall'][i]:.4f}, "
        f"F1={reduced_cv['test_f1'][i]:.4f}, "
        f"ROC-AUC={reduced_cv['test_roc_auc'][i]:.4f}")

Reduced XGBoost fold-wise results:
Fold 1: Accuracy=0.7692, Precision=0.7778, Recall=0.6364, F1=0.7000, ROC-AUC=0.7288
Fold 2: Accuracy=0.7600, Precision=0.7143, Recall=0.7143, F1=0.7143, ROC-AUC=0.8030
Fold 3: Accuracy=0.6538, Precision=0.6111, Recall=0.5000, F1=0.5500, ROC-AUC=0.7530
Fold 4: Accuracy=0.6471, Precision=0.6000, Recall=0.5455, F1=0.5714, ROC-AUC=0.7539


In [144]:
print("\nMean ± Std:")
for metric in scoring:
    scores = reduced_cv[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")


Mean ± Std:
accuracy: 0.7075 ± 0.0572
precision: 0.6758 ± 0.0738
recall: 0.5990 ± 0.0827
f1: 0.6339 ± 0.0738
roc_auc: 0.7597 ± 0.0269


In [156]:
from sklearn.metrics import average_precision_score,accuracy_score,roc_auc_score,confusion_matrix,classification_report

In [157]:
selected_model = clone(xgb_search.best_estimator_)

In [158]:
holdout_pred = selected_model.predict(X_holdout)
holdout_prob = selected_model.predict_proba(X_holdout)[:, 1]

NotFittedError: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
holdout_accuracy = accuracy_score(y_holdout, holdout_pred)
holdout_precision = precision_score(y_holdout, holdout_pred, zero_division=0)
holdout_recall = recall_score(y_holdout, holdout_pred, zero_division=0)
holdout_f1 = f1_score(y_holdout, holdout_pred, zero_division=0)
holdout_roc_auc = roc_auc_score(y_holdout, holdout_prob)
holdout_ap = average_precision_score(y_holdout, holdout_prob)

In [ ]:
print("FINAL HOLDOUT RESULTS")
print("---------------------")
print(f"Accuracy:           {holdout_accuracy:.4f}")
print(f"Precision:          {holdout_precision:.4f}")
print(f"Recall:             {holdout_recall:.4f}")
print(f"F1:                 {holdout_f1:.4f}")
print(f"ROC-AUC:            {holdout_roc_auc:.4f}")
print(f"Average Precision:  {holdout_ap:.4f}")

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_holdout, holdout_pred))

print("\nClassification Report:")
print( classification_report(y_holdout,holdout_pred, target_names=['Permanent', 'Contract'],zero_division=0))

In [ ]:
# Verify that the final preprocessing pipeline was fitted on all 205 development rows

numeric_transformer = (selected_model.named_steps['preprocessor']
    .named_transformers_['numeric'])

scaler = numeric_transformer.named_steps['scaler']

print("Scaler samples seen:", scaler.n_samples_seen_)
print("Expected development rows:", X.shape[0])
print("Holdout rows:", X_holdout.shape[0])

In [ ]:
import numpy as np

In [ ]:
rng = np.random.RandomState(42)

In [ ]:
n_bootstrap = 5000

In [ ]:
holdout_results = pd.DataFrame({
    'company_name': holdout_data['company_name'].values,
    'y_true': y_holdout.values,
    'y_pred': holdout_pred,
    'y_prob': holdout_prob})

In [ ]:
companies = holdout_results['company_name'].unique()

In [ ]:
bootstrap_results = { 'accuracy': [],'precision': [],
    'recall': [], 'f1': [], 'roc_auc': []}

In [ ]:
for _ in range(n_bootstrap):

    sampled_companies = rng.choice(companies,size=len(companies),replace=True)

    sampled_parts = [holdout_results[holdout_results['company_name'] == company]
        for company in sampled_companies
    ]

    sample = pd.concat( sampled_parts,ignore_index=True )

    y_true_sample = sample['y_true']
    y_pred_sample = sample['y_pred']
    y_prob_sample = sample['y_prob']

    bootstrap_results['accuracy'].append( accuracy_score(y_true_sample, y_pred_sample))

    bootstrap_results['precision'].append( precision_score( y_true_sample, y_pred_sample,zero_division=0))

    bootstrap_results['recall'].append(recall_score(y_true_sample,y_pred_sample,zero_division=0))

    bootstrap_results['f1'].append(f1_score(y_true_sample, y_pred_sample, zero_division=0 ))

    if y_true_sample.nunique() == 2:
        bootstrap_results['roc_auc'].append(roc_auc_score( y_true_sample, y_prob_sample))

In [ ]:
print("95% Company-Cluster Bootstrap Confidence Intervals")
print("---------------------------------------------------")

for metric, values in bootstrap_results.items():

    values = np.array(values)

    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)

    print(f"{metric}: "f"{np.mean(values):.4f} "f"(95% CI: {lower:.4f} - {upper:.4f})" )

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plot_data = (permutation_summary.sort_values('mean', ascending=True))
plt.figure(figsize=(10, 7))
plt.barh(plot_data.index,plot_data['mean'],xerr=plot_data['std'],capsize=4)
plt.xlabel("Mean Decrease in F1")
plt.ylabel("Feature")
plt.title("ML- Group-Aware Permutation Importance")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions( y_holdout, holdout_pred, display_labels=['Permanent', 'Contract'])
plt.title("ML-3 - Final Holdout Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay


RocCurveDisplay.from_predictions( y_holdout, holdout_prob)
plt.title("ML- Final Holdout ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_predictions(y_holdout,holdout_prob)
plt.title("ML-3 - Final Holdout Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.show()

In [ ]:
model_names = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']

f1_means = [ 0.6425,0.5915,0.6329, 0.6560]

f1_stds = [ 0.0897,0.0820,0.0493, 0.0775]

plt.figure(figsize=(9, 5))
plt.bar( model_names,f1_means, yerr=f1_stds,capsize=5)
plt.ylabel("F1-score")
plt.xlabel("Model")
plt.title("ML Tuned Model Comparison by F1")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
auc_means = [ 0.7940,0.7092, 0.7537, 0.7648]
auc_stds = [ 0.0466,0.0858, 0.0292, 0.0284]

plt.figure(figsize=(9, 5))
plt.bar( model_names, auc_means, yerr=auc_stds,capsize=5)
plt.ylabel("ROC-AUC")
plt.xlabel("Model")
plt.title("ML-Tuned Model Comparison by ROC-AUC")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [154]:
import joblib

from sklearn.base import clone
from sklearn.utils.validation import check_is_fitted


# =========================================================
# FINAL ML-3 MODEL
# =========================================================

# Create a completely fresh copy of the selected pipeline.
# clone() removes any old fitted/unfitted state but keeps
# the selected hyperparameters.
final_model = clone(  xgb_search.best_estimator_)


# Make sure we are using the complete development data
print("Development X shape:", X.shape)
print("Development y shape:", y.shape)

assert len(X) == 205, ( f"Expected 205 development rows, got {len(X)}")

assert len(y) == 205, ( f"Expected 205 target rows, got {len(y)}")


# Fit ONLY on the development data
# Do NOT use X_holdout or y_holdout here.
final_model.fit(X,y)


# Verify that the pipeline is actually fitted
check_is_fitted(final_model)

print("\nFinal ML-3 model fitted successfully.")
print("Training rows:", len(X))
print("Training companies:", dev_data["company_name"].nunique())

Development X shape: (205, 16)
Development y shape: (205,)

Final ML-3 model fitted successfully.
Training rows: 205
Training companies: 151


In [159]:
# =========================================================
# FINAL HOLDOUT PREDICTIONS
# =========================================================

X_holdout_final = X_holdout[core_features].copy()

holdout_prob = selected_model.predict_proba(
    X_holdout_final)[:, 1]

holdout_pred = (holdout_prob >= threshold).astype(int)

print("Holdout predictions generated successfully.")

NotFittedError: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
import joblib

model_package = { 'model': selected_model,'features': core_features,
    'threshold': 0.50,'target_mapping': {'permanent': 0,'contract': 1 }}

joblib.dump(model_package, "contract_type_xgb_model_final.pkl")

print("Corrected ML model saved successfully.")

In [151]:
import joblib

# Load the final fitted ML-3 model
model_package = joblib.load("contract_type_xgb_model_final.pkl")

# Extract the fitted pipeline
selected_model = model_package["model"]

# Extract saved threshold
threshold = model_package["threshold"]

print("Final ML-3 model loaded.")
print("Threshold:", threshold)

Final ML-3 model loaded.
Threshold: 0.5


## Final Model Refit and Save

After model selection and feature analysis, the selected XGBoost configuration was refitted using the complete development dataset and the final 16-feature set.

The final model uses:

- **Model:** XGBoost Classifier
- **Features:** 16
- **Classification threshold:** 0.50
- **Target mapping:** Permanent = 0, Contract = 1

The preprocessing steps are retained inside the fitted pipeline so that the same transformations are applied during prediction.

The fitted model and its metadata were saved as:

`contract_type_xgb_model_final.pkl`

In [160]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

categorical_features = ['location_clean', 'category','standard_role','contract_time']

numeric_features = ['posting_year','posting_month','title_word_count','title_length',
    'title_has_senior','title_has_lead','title_has_manager','title_has_intern',
    'title_has_junior','title_has_principal', 'title_has_staff', 'title_has_associate']


categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant',fill_value='Missing' )),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median' ))])

preprocessor = ColumnTransformer([
    ('categorical', categorical_pipeline,categorical_features),
    ('numeric',numeric_pipeline, numeric_features )])

final_xgb = XGBClassifier(n_estimators=400,max_depth=2,learning_rate=0.1,subsample=0.8,
    colsample_bytree=0.8,min_child_weight=1,gamma=0.1,reg_alpha=0.1,reg_lambda=2,
    objective='binary:logistic',eval_metric='logloss',random_state=42, n_jobs=-1)

final_pipeline = Pipeline([
    ( 'preprocessor',preprocessor),
    ('model',final_xgb )])

In [161]:
# =========================================================
# SAVE FINAL ML-3 MODEL
# =========================================================

core_features = ["location_clean","category","standard_role", "contract_time",
    "posting_year","posting_month","title_word_count","title_length","title_has_senior",
    "title_has_lead", "title_has_manager","title_has_intern", "title_has_junior",
    "title_has_principal","title_has_staff", "title_has_associate"]

# Important:
# Your final model currently uses the 16-feature version,
# not the reduced 9-feature experiment.

threshold = 0.50

target_mapping = {"permanent": 0, "contract": 1}

model_package = { "model": final_model,"features": core_features,
    "threshold": threshold,"target_mapping": target_mapping}

joblib.dump(model_package,"contract_type_xgb_model_final.pkl")

print("\nCorrected model saved as "
    "contract_type_xgb_model_final.pkl")


Corrected model saved as contract_type_xgb_model_final.pkl


In [162]:
# =========================================================
# VERIFY SAVED MODEL
# =========================================================

saved_package = joblib.load("contract_type_xgb_model_final.pkl")

saved_model = saved_package["model"]

check_is_fitted(saved_model)

print("Saved model is fitted successfully.")
print("Saved threshold:", saved_package["threshold"])
print("Saved feature count:", len(saved_package["features"]))

Saved model is fitted successfully.
Saved threshold: 0.5
Saved feature count: 16


## Final Holdout Evaluation

The final holdout dataset was kept completely separate from model development, feature selection, and hyperparameter tuning.

The refitted final model was applied to the holdout set to generate:

- predicted contract classes
- probability estimates for the positive class (`contract`)

The predefined classification threshold of **0.50** was used to convert predicted probabilities into final contract-type classes.

The resulting predictions were then used to calculate the final holdout performance metrics.

In [163]:
# =========================================================
# FINAL HOLDOUT PREDICTIONS
# =========================================================

X_holdout_final = X_holdout[core_features].copy()

holdout_prob = saved_model.predict_proba( X_holdout_final)[:, 1]

holdout_pred = (holdout_prob >= threshold).astype(int)

print("Holdout predictions generated successfully.")

Holdout predictions generated successfully.


In [165]:
holdout_accuracy = accuracy_score(y_holdout, holdout_pred)
holdout_precision = precision_score(y_holdout, holdout_pred, zero_division=0)
holdout_recall = recall_score(y_holdout, holdout_pred, zero_division=0)
holdout_f1 = f1_score(y_holdout, holdout_pred, zero_division=0)
holdout_roc_auc = roc_auc_score(y_holdout, holdout_prob)
holdout_ap = average_precision_score(y_holdout, holdout_prob)

In [166]:
print("FINAL HOLDOUT RESULTS")
print("---------------------")
print(f"Accuracy:           {holdout_accuracy:.4f}")
print(f"Precision:          {holdout_precision:.4f}")
print(f"Recall:             {holdout_recall:.4f}")
print(f"F1:                 {holdout_f1:.4f}")
print(f"ROC-AUC:            {holdout_roc_auc:.4f}")
print(f"Average Precision:  {holdout_ap:.4f}")

FINAL HOLDOUT RESULTS
---------------------
Accuracy:           0.6667
Precision:          0.6667
Recall:             0.4545
F1:                 0.5405
ROC-AUC:            0.7069
Average Precision:  0.6704


## Final Holdout Results

The final holdout evaluation produced the following results:

- **Accuracy:** 0.6667
- **Precision:** 0.6667
- **Recall:** 0.4545
- **F1-score:** 0.5405
- **ROC-AUC:** 0.7069
- **Average Precision:** 0.6704

These results indicate **modest predictive performance** on previously unseen companies. The relatively limited holdout size and the small number of labeled contract-type observations mean that the estimates should be interpreted with caution.

Overall, the model is best treated as an **experimental contract-type predictor** rather than a production-ready model.

# ML: Contract Type Prediction — Final Conclusion

## 1. Objective

The objective of ML was to predict the `contract_type` of a job posting as either:

- `permanent`
- `contract`

The positive class used for evaluation was `contract`, encoded as `1`, while `permanent` was encoded as `0`.

---

## 2. Dataset

The complete job-market dataset contained 7,269 job postings.

However, `contract_type` was available for only 256 postings:

- Permanent: 147 (57.42%)
- Contract: 109 (42.58%)

All 256 labeled observations came from the Adzuna source and represented 187 companies.

The company structure was highly concentrated with respect to the target: only 3 companies had both `contract` and `permanent` postings, while the remaining labeled companies contained only one observed contract type.

This company-level structure was important for the evaluation design because a random row-level split could place postings from the same company in both development and validation data.

---

## 3. Evaluation Design

To reduce company-level leakage, a company-aware evaluation strategy was used.

`StratifiedGroupKFold` was used so that:

- class proportions were approximately preserved across folds, and
- postings from the same company could not appear in both training and validation portions of a fold.

A company-separated final holdout was established before model selection:

- Development set: 205 postings
- Development companies: 151
- Final holdout: 51 postings
- Holdout companies: 36
- Company overlap: 0

The development set contained:

- Permanent: 118 (57.56%)
- Contract: 87 (42.44%)

The holdout contained:

- Permanent: 29 (56.86%)
- Contract: 22 (43.14%)

The 51-row holdout was kept untouched during feature analysis, model comparison, hyperparameter tuning, and feature ablation.

---

## 4. Feature Selection

The final feature set contained 16 features:

- `location_clean`
- `category`
- `standard_role`
- `contract_time`
- `posting_year`
- `posting_month`
- `title_word_count`
- `title_length`
- `title_has_senior`
- `title_has_lead`
- `title_has_manager`
- `title_has_intern`
- `title_has_junior`
- `title_has_principal`
- `title_has_staff`
- `title_has_associate`

The following variables were excluded from the final feature set:

- `contract_type` — target variable
- `company_name` — excluded from predictors because of strong company-level target concentration
- `source` — constant within the labeled modeling population
- `job_level`
- `job_type`
- `job_level_clean` — unavailable in the contract-type labeled subset

The raw `title` column was not used directly; title-derived features such as title length, word count, and selected indicator variables were used instead.

Missing values in `contract_time` were retained as a separate `Missing` category rather than removing those observations.

---

## 5. Preprocessing

The preprocessing pipeline was fitted within the model pipeline to avoid fitting preprocessing transformations on validation data.

Categorical features were processed using:

- constant imputation with `Missing`
- one-hot encoding
- `handle_unknown='ignore'`

Numeric features were processed using:

- median imputation
- standard scaling

This preprocessing was applied separately within the cross-validation training folds.

---

## 6. Models Evaluated

The following approaches were evaluated:

1. Dummy Classifier
2. Logistic Regression
3. Decision Tree
4. Random Forest
5. XGBoost

The development-stage models were evaluated using company-aware cross-validation.

Hyperparameter tuning was performed using the development data and a 4-fold `StratifiedGroupKFold`.

F1-score was used as the predefined hyperparameter-selection metric, while Accuracy, Precision, Recall, F1-score, and ROC-AUC were also recorded.

---

## 7. Tuned Model Results

The selected tuned configurations produced the following inner cross-validation summaries:

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.6986 ± 0.0795 | 0.6550 ± 0.0929 | 0.6461 ± 0.1314 | 0.6425 ± 0.0897 | 0.7940 ± 0.0466 |
| Decision Tree | 0.6923 ± 0.0689 | 0.7058 ± 0.1451 | 0.5281 ± 0.1010 | 0.5915 ± 0.0820 | 0.7092 ± 0.0858 |
| Random Forest | 0.6976 ± 0.0547 | 0.6654 ± 0.0872 | 0.6098 ± 0.0459 | 0.6329 ± 0.0493 | 0.7537 ± 0.0292 |
| XGBoost | 0.7319 ± 0.0488 | 0.7128 ± 0.0599 | 0.6098 ± 0.0911 | 0.6560 ± 0.0775 | 0.7648 ± 0.0284 |

The tuned XGBoost configuration had the highest observed F1 among these tuned configurations, with an inner-CV F1 of `0.6560 ± 0.0775`.

These inner-CV values should be interpreted as development/model-selection results rather than as independent estimates of future performance because the hyperparameters were selected using the same inner-CV process.

---

## 8. Selected XGBoost Model

The selected XGBoost hyperparameters were:

- `n_estimators = 400`
- `max_depth = 2`
- `learning_rate = 0.10`
- `subsample = 0.80`
- `colsample_bytree = 0.80`
- `min_child_weight = 1`
- `gamma = 0.1`
- `reg_alpha = 0.1`
- `reg_lambda = 2`

After model selection, a fresh copy of the selected pipeline was fitted on all 205 development observations.

The fitted model therefore used:

- 205 development postings
- 151 development companies
- all 16 final features

The final 51-row holdout was then evaluated once.

---

## 9. Feature Importance

Two feature-importance approaches were examined.

### XGBoost Gain-Based Importance

The largest aggregated feature contributions in the fitted XGBoost model were:

| Feature | Importance |
|---|---:|
| `standard_role` | 0.4570 |
| `location_clean` | 0.2142 |
| `contract_time` | 0.0798 |
| `category` | 0.0583 |
| `posting_year` | 0.0501 |
| `posting_month` | 0.0433 |
| `title_has_senior` | 0.0339 |
| `title_length` | 0.0325 |
| `title_word_count` | 0.0309 |

The remaining title-indicator features had zero gain-based importance in this fitted model.

### Group-Aware Permutation Importance

Group-aware permutation importance produced the following average impacts on validation F1:

| Feature | Mean | Std |
|---|---:|---:|
| `posting_month` | 0.1042 | 0.0575 |
| `location_clean` | 0.0910 | 0.0157 |
| `contract_time` | 0.0655 | 0.0684 |
| `title_length` | 0.0606 | 0.0686 |
| `posting_year` | 0.0560 | 0.0623 |
| `standard_role` | 0.0389 | 0.0727 |
| `title_word_count` | 0.0319 | 0.0319 |
| `category` | 0.0068 | 0.0048 |
| `title_has_senior` | 0.0041 | 0.0015 |

The two importance methods produced different rankings. This indicates that feature importance is sensitive to the interpretation method and should not be treated as causal evidence.

The relatively high importance of `posting_month`, `posting_year`, and missingness in `contract_time` was treated cautiously because these variables may contain temporal, collection, or labeling-process information rather than stable employment relationships.

---

## 10. Feature Ablation

Seven features had zero observed importance in the fitted XGBoost model:

- `title_has_associate`
- `title_has_intern`
- `title_has_junior`
- `title_has_lead`
- `title_has_manager`
- `title_has_principal`
- `title_has_staff`

A reduced 9-feature XGBoost model was tested using the same selected XGBoost hyperparameters.

Results:

| Metric | Full 16 Features | Reduced 9 Features |
|---|---:|---:|
| Accuracy | 0.7319 | 0.7075 |
| Precision | 0.7128 | 0.6758 |
| Recall | 0.6098 | 0.5990 |
| F1 | 0.6560 | 0.6339 |
| ROC-AUC | 0.7648 | 0.7597 |

The reduced feature set did not improve the development results. Therefore, the original 16-feature set was retained.

---

## 11. Classification Threshold

The final holdout was evaluated using the predefined classification threshold of `0.50`.

The threshold was not optimized using the final holdout.

Therefore, the final holdout results represent performance at a fixed threshold of 0.50 rather than performance after holdout-specific threshold tuning.

The Precision-Recall analysis was also examined because recall for the `contract` class was relatively low.

---

## 12. Final Holdout Evaluation

The selected XGBoost model was evaluated once on the untouched 51-row holdout containing previously unseen companies.

Final holdout results:

| Metric | Final Holdout |
|---|---:|
| Accuracy | **0.6667** |
| Precision | **0.6667** |
| Recall | **0.4545** |
| F1-score | **0.5405** |
| ROC-AUC | **0.7069** |
| Average Precision | **0.6704** |

Confusion matrix:

| Actual / Predicted | Permanent | Contract |
|---|---:|---:|
| Permanent | 24 | 5 |
| Contract | 12 | 10 |

This means:

- 24 permanent postings were correctly classified
- 5 permanent postings were classified as contract
- 10 contract postings were correctly classified
- 12 contract postings were classified as permanent

For the positive class `contract`, the model correctly identified 10 of the 22 contract postings in the final holdout.

---

## 13. Uncertainty Analysis

Because the final holdout contained only 51 postings from 36 companies, a company-cluster bootstrap with 5,000 resamples was used to quantify uncertainty.

The 95% company-cluster bootstrap confidence intervals were:

| Metric | Point Estimate | 95% CI |
|---|---:|---:|
| Accuracy | 0.6667 | 0.4773 – 0.8222 |
| Precision | 0.6667 | 0.3846 – 0.9000 |
| Recall | 0.4545 | 0.2273 – 0.6875 |
| F1-score | 0.5405 | 0.2963 – 0.7308 |
| ROC-AUC | 0.7069 | 0.5298 – 0.8549 |

The wide intervals indicate substantial uncertainty due to the small number of holdout observations and companies.

---

## 14. Final Interpretation

The ML-3 experiment shows that the available job-posting features contain predictive information for distinguishing between `contract` and `permanent` postings when evaluated across previously unseen companies.

The selected XGBoost model achieved:

- **66.67% accuracy**
- **66.67% precision**
- **45.45% recall**
- **54.05% F1-score**
- **0.7069 ROC-AUC**
- **0.6704 Average Precision**

on the final company-separated holdout.

The model performed above the majority-class accuracy baseline of 56.86%, but the recall for the `contract` class was 45.45%, meaning that 12 of the 22 actual contract postings in the holdout were not identified as contract.

The difference between the development cross-validation results and the final holdout results also shows that performance estimates from the small development sample should not be assumed to transfer directly to unseen companies.

The feature analysis indicates that `posting_month`, `location_clean`, `contract_time`, `title_length`, `posting_year`, and `standard_role` were among the features associated with changes in validation F1 under the group-aware permutation analysis. These relationships should be interpreted as model behavior rather than as causal effects.

---

## 15. Limitations

The main limitations of ML-3 are:

1. Only 256 of 7,269 postings had `contract_type` labels, approximately 3.52% of the complete dataset.

2. All labeled observations came from Adzuna, so the model was not directly validated on Greenhouse contract-type labels.

3. The labeled data were concentrated at the company level, with only three companies containing both observed target classes.

4. The final holdout contained only 51 postings from 36 companies, producing wide confidence intervals.

5. Temporal variables such as `posting_year` and `posting_month` showed meaningful importance and may partly reflect collection or batch effects.

6. The final threshold of 0.50 was predefined rather than optimized within the corrected evaluation workflow.

7. The reported final performance should therefore not be generalized directly to all 7,269 postings, all companies, or other job sources.

---

## 16. Final ML-3 Result

**Model:** Tuned XGBoost Classifier

**Feature set:** 16 features

**Classification threshold:** 0.50

**Final evaluation:** 51 postings from 36 previously unseen companies

**Company overlap:** 0

**Final holdout performance:**

- Accuracy = **0.6667**
- Precision = **0.6667**
- Recall = **0.4545**
- F1-score = **0.5405**
- ROC-AUC = **0.7069**
- Average Precision = **0.6704**

Overall, ML-3 provides evidence of **modest predictive signal for contract-type classification across unseen companies**, but the small labeled dataset, limited holdout size, company-level concentration, and wide uncertainty intervals mean that the model should be treated as an experimental predictor rather than a broadly validated production model.